In [1]:
import requests
import timeit
import pandas as pd
import redis

In [2]:
url = "http://redis-app-1:5000"
r = redis.Redis(host='redis', port=6379, db=0)

In [3]:
for i in range(3):
    resposta = requests.get(f"{url}/")
    i+1
    print(f"{resposta.text}")

Hello World! I have been seen 1 times.
Tempo de resposta: 0.001359 segundos
Hello World! I have been seen 2 times.
Tempo de resposta: 0.000444 segundos
Hello World! I have been seen 3 times.
Tempo de resposta: 0.000347 segundos


In [4]:
resposta_set = requests.get(f"{url}/set/cursos/Redis")
print(resposta_set.text)

Valor Redis foi armazenado com a chave cursos


In [5]:
resposta_get = requests.get(f"{url}/get/cursos")
print(resposta_get.text)

Valor recuperado: Redis


In [6]:
def fetch_sem_cache():
    requests.get(f"{url}/fib/30")

def fetch_com_cache():
    requests.get(f"{url}/fib-cache/30")

In [7]:
tempo_sem_cache = timeit.timeit(fetch_sem_cache, number=1)
r.delete("fib:30")
tempo_cache_miss = timeit.timeit(fetch_com_cache, number=1)

numero_execucoes = 10
tempo_total_hit = timeit.timeit(fetch_com_cache, number=numero_execucoes)
tempo_medio_hit = tempo_total_hit / numero_execucoes

In [8]:
resultados = {
    "Cenário": ["Sem Cache", "Com Cache (Miss)", "Com Cache (Hit - Média)"],
    "Tempo (ms)": [
        tempo_sem_cache * 1000, 
        tempo_cache_miss * 1000, 
        tempo_medio_hit * 1000
    ]
}

df_tempos = pd.DataFrame(resultados)
display(df_tempos)

ttl_restante = r.ttl("fib:30")
print(f"\nTempo de vida restante (TTL) da chave 'fib:30': {ttl_restante} segundos")

,Cenário,Tempo (ms)
0,Sem Cache,117.199131
1,Com Cache (Miss),113.021231
2,Com Cache (Hit - Média),3.250584



Tempo de vida restante (TTL) da chave 'fib:30': 60 segundos


Quando a operação é feito do zero, o tempo é relativamente alto e semelhante em 'sem cache' e 'com cache'. Porém quando o operação já está em cache e somente resgata o resultado, o tempo cai drasticamente.